# 01. Dataset preprocessing

Preprocesamiento reproducible de las características radiómicas antes de modelar. Esta primera versión aplica:

1. imputación por mediana de valores faltantes (incluidos infinitos),
2. winsorización por percentiles, y
3. eliminación de características con correlación alta.

La columna `target` y el identificador de imagen se conservan sin modificación.

> **Nota para modelado:** este notebook prepara un dataset canónico y calcula los parámetros con todos los datos. Al evaluar modelos, las medianas, límites de winsorización y filtro de correlación deberán ajustarse exclusivamente con el conjunto de entrenamiento de cada partición para evitar fuga de información.

## 0. Setup

In [1]:
import sys
sys.path.insert(0, "../..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl

from src.paths import DATA_DIR, PROCESSED_DIR, RESULTS_DIR, ensure_dir

INPUT_CSV = DATA_DIR / "features_features_all_channels_target.csv"
METADATA_PARQUET = PROCESSED_DIR / "isic_selected.parquet"
OUTPUT_CSV = PROCESSED_DIR / "features_clean.csv"
DROPPED_FEATURES_CSV = PROCESSED_DIR / "features_dropped.csv"
OUT_DIR = ensure_dir(RESULTS_DIR / "eda" / "preprocessing")

ID_COL = "image_id"
TARGET_COL = "target"

# Hiperparámetros iniciales del preprocesamiento.
WINSOR_LOW = 0.01
WINSOR_HIGH = 0.99
CORR_THRESHOLD = 0.95

print(f"Input:  {INPUT_CSV}")
print(f"Metadata: {METADATA_PARQUET}")
print(f"Output: {OUTPUT_CSV}")
print(f"Winsorización: p{WINSOR_LOW:.0%}–p{WINSOR_HIGH:.0%}")
print(f"Umbral de correlación: |r| > {CORR_THRESHOLD}")

Input:  /Users/indalecio.rios/Documents/personal-github/scd-ml/data/features_features_all_channels_target.csv
Metadata: /Users/indalecio.rios/Documents/personal-github/scd-ml/results/processed/isic_selected.parquet
Output: /Users/indalecio.rios/Documents/personal-github/scd-ml/results/processed/features_clean.csv
Winsorización: p1%–p99%
Umbral de correlación: |r| > 0.95


## 1. Carga y validación

Solo se transforman las columnas radiómicas; `image_id`, `target` y cualquier otra columna de metadatos se preservan.

In [2]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"No se encontró {INPUT_CSV}. Ejecuta primero la última celda de "
        "'01 ISIC archive - Features.ipynb' para generar el CSV con target."
    )

df = pd.read_csv(INPUT_CSV)
required_cols = {ID_COL, TARGET_COL}
missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(f"Faltan columnas requeridas: {sorted(missing_required)}")

if not METADATA_PARQUET.exists():
    raise FileNotFoundError(f"No se encontró la metadata: {METADATA_PARQUET}")

meta = pd.DataFrame(pl.read_parquet(METADATA_PARQUET).to_dicts())
meta_id_col = "isic_id" if "isic_id" in meta.columns else ID_COL
if meta_id_col not in meta.columns:
    raise ValueError("La metadata no contiene un identificador compatible con image_id.")

anatomy_cols = sorted(c for c in meta.columns if c.startswith("anatom_site_"))
clinical_base_cols = ["age_approx", "pixels_x", "pixels_y", "sex"]
clinical_cols = [c for c in clinical_base_cols if c in meta.columns] + anatomy_cols
required_clinical = {"age_approx", "pixels_x", "pixels_y", "sex", "anatom_site_unknown"}
missing_clinical = required_clinical - set(clinical_cols)
if missing_clinical:
    raise ValueError(f"Faltan columnas clínicas requeridas: {sorted(missing_clinical)}")
if meta[meta_id_col].duplicated().any():
    raise ValueError("La metadata contiene identificadores duplicados.")

df = df.merge(
    meta[[meta_id_col, *clinical_cols]].rename(columns={meta_id_col: ID_COL}),
    on=ID_COL,
    how="left",
    validate="one_to_one",
)
n_without_metadata = int(df[clinical_cols].isna().all(axis=1).sum())
print(f"Filas sin metadata clínica asociada: {n_without_metadata:,}")

feature_cols = [c for c in df.columns if "__original_" in c]
if not feature_cols:
    raise ValueError("No se detectaron columnas de características radiómicas.")

metadata_cols = [c for c in df.columns if c not in feature_cols]
print(f"Dataset: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")
print(f"Características radiómicas: {len(feature_cols):,}")
print(f"Metadata clínica incorporada: {clinical_cols}")
print("\nDistribución del target:")
print(df[TARGET_COL].value_counts(dropna=False))

Filas sin metadata clínica asociada: 0
Dataset: 76,715 filas × 420 columnas
Características radiómicas: 408
Metadata clínica incorporada: ['age_approx', 'pixels_x', 'pixels_y', 'sex', 'anatom_site_Anogenital region', 'anatom_site_Head and neck', 'anatom_site_Lower extremity', 'anatom_site_Trunk', 'anatom_site_Upper extremity', 'anatom_site_unknown']

Distribución del target:
target
Benign-melanocytic           48912
Malignant-melanocytic        10610
Benign-non-melanocytic        9559
Malignant-non-melanocytic     7634
Name: count, dtype: int64


## 2. Auditoría e imputación de metadata clínica

La edad y el sexo ya fueron imputados en el EDA de imágenes. Esta etapa los audita y aplica una imputación de respaldo independiente del `target`: mediana global para edad y dimensiones, moda global para sexo y la categoría `unknown` para posiciones anatómicas inválidas o faltantes.

In [3]:
clinical = df[clinical_cols].copy()
clinical_missing_before = clinical.isna().sum()

numeric_clinical_cols = [c for c in ["age_approx", "pixels_x", "pixels_y"] if c in clinical]
for col in numeric_clinical_cols:
    clinical[col] = pd.to_numeric(clinical[col], errors="coerce")
    median = clinical[col].median()
    if pd.isna(median):
        raise ValueError(f"La columna clínica '{col}' está completamente vacía.")
    clinical[col] = clinical[col].fillna(median)

sex_mode = clinical["sex"].dropna().mode()
sex_fill_value = sex_mode.iloc[0] if not sex_mode.empty else "unknown"
clinical["sex"] = clinical["sex"].fillna(sex_fill_value)

anatomy_values = clinical[anatomy_cols].apply(pd.to_numeric, errors="coerce")
invalid_anatomy = anatomy_values.isna().any(axis=1) | anatomy_values.sum(axis=1).ne(1)
clinical.loc[invalid_anatomy, anatomy_cols] = 0
clinical.loc[invalid_anatomy, "anatom_site_unknown"] = 1
clinical[anatomy_cols] = clinical[anatomy_cols].astype("uint8")

clinical_missing_after = int(clinical.isna().sum().sum())
assert clinical_missing_after == 0, f"Quedaron {clinical_missing_after} valores faltantes en metadata"

clinical_report = pd.DataFrame({
    "column": clinical_cols,
    "missing_before": clinical_missing_before.reindex(clinical_cols).values,
    "missing_after": clinical.isna().sum().reindex(clinical_cols).values,
})
print(f"Registros con posición anatómica reasignada a unknown: {int(invalid_anatomy.sum()):,}")
clinical_report

Registros con posición anatómica reasignada a unknown: 0


,column,missing_before,missing_after
0,age_approx,0,0
1,pixels_x,0,0
2,pixels_y,0,0
3,sex,0,0
4,anatom_site_Anogenital region,0,0
5,anatom_site_Head and neck,0,0
6,anatom_site_Lower extremity,0,0
7,anatom_site_Trunk,0,0
8,anatom_site_Upper extremity,0,0
9,anatom_site_unknown,0,0


## 3. Imputación de valores faltantes en características radiómicas

Los valores `+/-inf` se convierten primero a faltantes. Cada característica se imputa con su mediana calculada sobre el conjunto completo. Las columnas completamente faltantes se eliminan porque no tienen una mediana definible.

In [4]:
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()

missing_before = X.isna().sum()
all_nan_cols = missing_before[missing_before == len(X)].index.tolist()
missing_report = pd.DataFrame({
    "feature": X.columns,
    "missing_before": missing_before.values,
    "missing_pct_before": (missing_before.values / len(X) * 100).round(3),
}).sort_values("missing_before", ascending=False)

print(f"Valores faltantes antes de imputar: {int(missing_before.sum()):,}")
print(f"Columnas completamente faltantes: {len(all_nan_cols):,}")

X = X.drop(columns=all_nan_cols)
medians = X.median(axis=0)
X_imputed = X.fillna(medians)

remaining_missing = int(X_imputed.isna().sum().sum())
assert remaining_missing == 0, f"Quedaron {remaining_missing:,} valores faltantes"
print(f"Valores faltantes después de imputar: {remaining_missing:,}")
print(f"Características disponibles: {X_imputed.shape[1]:,}")

missing_report.head(15)

Valores faltantes antes de imputar: 0
Columnas completamente faltantes: 0
Valores faltantes después de imputar: 0
Características disponibles: 408


,feature,missing_before,missing_pct_before
0,blue__original_shape2D_Elongation,0,0.0
268,red__original_gldm_SmallDependenceLowGrayLevel...,0,0.0
278,red__original_glrlm_RunLengthNonUniformity,0,0.0
277,red__original_glrlm_RunEntropy,0,0.0
276,red__original_glrlm_LowGrayLevelRunEmphasis,0,0.0
275,red__original_glrlm_LongRunLowGrayLevelEmphasis,0,0.0
274,red__original_glrlm_LongRunHighGrayLevelEmphasis,0,0.0
273,red__original_glrlm_LongRunEmphasis,0,0.0
272,red__original_glrlm_HighGrayLevelRunEmphasis,0,0.0
271,red__original_glrlm_GrayLevelVariance,0,0.0


## 4. Winsorización de valores atípicos

Cada característica queda acotada a sus percentiles p1 y p99. Los límites se calculan después de imputar.

In [5]:
lower_bounds = X_imputed.quantile(WINSOR_LOW)
upper_bounds = X_imputed.quantile(WINSOR_HIGH)

n_clipped_low = int(X_imputed.lt(lower_bounds, axis=1).sum().sum())
n_clipped_high = int(X_imputed.gt(upper_bounds, axis=1).sum().sum())
X_winsor = X_imputed.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

print(f"Valores recortados por debajo de p{WINSOR_LOW:.0%}: {n_clipped_low:,}")
print(f"Valores recortados por encima de p{WINSOR_HIGH:.0%}: {n_clipped_high:,}")
print(f"Total recortado: {n_clipped_low + n_clipped_high:,} "
      f"({(n_clipped_low + n_clipped_high) / X_imputed.size:.2%} de las celdas)")

winsor_bounds = pd.DataFrame({
    "feature": X_winsor.columns,
    "lower_bound": lower_bounds.values,
    "upper_bound": upper_bounds.values,
})
winsor_bounds.head()

Valores recortados por debajo de p1%: 308,238
Valores recortados por encima de p99%: 306,393
Total recortado: 614,631 (1.96% de las celdas)


,feature,lower_bound,upper_bound
0,blue__original_shape2D_Elongation,0.451685,9.790547e-01
1,blue__original_shape2D_MajorAxisLength,113.858024,6.791394e+03
2,blue__original_shape2D_MaximumDiameter,116.275535,6.974482e+03
3,blue__original_shape2D_MeshSurface,7816.040000,2.277785e+07
4,blue__original_shape2D_MinorAxisLength,86.458327,4.541504e+03


## 5. Filtro de correlación

Se usa la correlación de Pearson absoluta. Para cada grupo de variables relacionadas con `|r| > 0.95`, se elimina iterativamente la variable con mayor correlación media con el resto; los empates se resuelven por nombre para mantener el resultado reproducible.

In [6]:
corr = X_winsor.corr().abs()
active_features = list(X_winsor.columns)
corr_dropped = []

while True:
    active_corr = corr.loc[active_features, active_features]
    upper = active_corr.where(np.triu(np.ones(active_corr.shape), k=1).astype(bool))
    high_pairs = upper.stack()[lambda s: s > CORR_THRESHOLD]

    if high_pairs.empty:
        break

    involved = sorted(set(high_pairs.index.get_level_values(0)) | set(high_pairs.index.get_level_values(1)))
    redundancy = active_corr.loc[involved, involved].mean(axis=1)
    max_redundancy = redundancy.max()
    loser = sorted(redundancy[redundancy == max_redundancy].index)[0]

    related = high_pairs[
        (high_pairs.index.get_level_values(0) == loser)
        | (high_pairs.index.get_level_values(1) == loser)
    ]
    corr_dropped.append({
        "dropped_feature": loser,
        "reason": f"corr>{CORR_THRESHOLD}",
        "max_abs_correlation": float(related.max()),
        "n_high_corr_pairs_at_removal": int(len(related)),
    })
    active_features.remove(loser)

X_final = X_winsor[active_features]
print(f"Características antes del filtro: {X_winsor.shape[1]:,}")
print(f"Características eliminadas: {len(corr_dropped):,}")
print(f"Características retenidas: {X_final.shape[1]:,}")

correlation_report = pd.DataFrame(
    corr_dropped,
    columns=["dropped_feature", "reason", "max_abs_correlation", "n_high_corr_pairs_at_removal"],
).sort_values("max_abs_correlation", ascending=False)
correlation_report.head(15)

Características antes del filtro: 408
Características eliminadas: 227
Características retenidas: 181


,dropped_feature,reason,max_abs_correlation,n_high_corr_pairs_at_removal
113,blue__original_shape2D_PerimeterSurfaceRatio,corr>0.95,1.0,3
140,blue__original_glcm_JointAverage,corr>0.95,1.0,4
142,green__original_shape2D_PixelSurface,corr>0.95,1.0,1
65,green__original_firstorder_Energy,corr>0.95,1.0,7
25,blue__original_shape2D_MaximumDiameter,corr>0.95,1.0,15
52,gray__original_shape2D_MajorAxisLength,corr>0.95,1.0,2
27,gray__original_shape2D_MaximumDiameter,corr>0.95,1.0,14
46,green__original_shape2D_Perimeter,corr>0.95,1.0,5
29,green__original_shape2D_MaximumDiameter,corr>0.95,1.0,13
94,gray__original_glcm_SumAverage,corr>0.95,1.0,2


## 6. Ensamblado y guardado

El archivo final contiene `image_id`, las características radiómicas procesadas, cualquier metadato adicional y `target`. También se guarda un manifiesto de exclusiones.

In [7]:
clean = pd.concat(
    [
        df[[ID_COL]].reset_index(drop=True),
        X_final.reset_index(drop=True),
        clinical.reset_index(drop=True),
        df[[TARGET_COL]].reset_index(drop=True),
    ],
    axis=1,
)

dropped_manifest = pd.concat(
    [
        pd.DataFrame({
            "dropped_feature": all_nan_cols,
            "reason": "all_missing",
            "max_abs_correlation": np.nan,
            "n_high_corr_pairs_at_removal": np.nan,
        }),
        pd.DataFrame(corr_dropped),
    ],
    ignore_index=True,
)

assert clean[ID_COL].equals(df[ID_COL])
assert clean[TARGET_COL].equals(df[TARGET_COL])
assert int(clean[clinical_cols].isna().sum().sum()) == 0
assert int(clean[X_final.columns].isna().sum().sum()) == 0
assert int(np.isinf(clean[X_final.columns].to_numpy(dtype=float)).sum()) == 0

ensure_dir(PROCESSED_DIR)
clean.to_csv(OUTPUT_CSV, index=False)
dropped_manifest.to_csv(DROPPED_FEATURES_CSV, index=False)

summary = pd.DataFrame([
    {"step": "Entrada", "n_features": len(feature_cols), "n_clinical": len(clinical_cols)},
    {"step": "Tras imputación", "n_features": X_imputed.shape[1], "n_clinical": len(clinical_cols)},
    {"step": "Tras filtro de correlación", "n_features": X_final.shape[1], "n_clinical": len(clinical_cols)},
])

print(f"Dataset final: {clean.shape[0]:,} filas × {clean.shape[1]:,} columnas")
print(f"Guardado en: {OUTPUT_CSV}")
print(f"Manifiesto de exclusiones: {DROPPED_FEATURES_CSV}")
summary

Dataset final: 76,715 filas × 193 columnas
Guardado en: /Users/indalecio.rios/Documents/personal-github/scd-ml/results/processed/features_clean.csv
Manifiesto de exclusiones: /Users/indalecio.rios/Documents/personal-github/scd-ml/results/processed/features_dropped.csv


,step,n_features,n_clinical
0,Entrada,408,10
1,Tras imputación,408,10
2,Tras filtro de correlación,181,10
